In [ ]:
from music21 import converter
from crim_intervals.main_objs import importScore
import pandas as pd

: 

Define the file path.
Convert the mei file into score.

In [3]:
filepath = 'Bach_BWV_0772.mei'
score = converter.parse(str(filepath))

Analyse for the key of the score, with the confidence factor.

In [4]:
key_analysis = score.analyze("key")
analysis_dict = {
    "Key Name": str(key_analysis),
    "Confidence Factor": key_analysis.correlationCoefficient,
}
analysis_dict

{'Key Name': 'C major', 'Confidence Factor': 0.9450617236680577}

get_notes

In [5]:
piece = importScore(str(filepath))
nr = piece.notes()
nr = piece.numberParts(nr)
nr = piece.detailIndex(nr)

nr.to_csv(index=True)

'Measure,Beat,1,2\r\n1.0,1.0,Rest,Rest\r\n1.0,1.25,C4,\r\n1.0,1.5,D4,\r\n1.0,1.75,E4,\r\n1.0,2.0,F4,\r\n1.0,2.25,D4,\r\n1.0,2.5,E4,\r\n1.0,2.75,C4,\r\n1.0,3.0,G4,\r\n1.0,3.25,,C3\r\n1.0,3.5,C5,D3\r\n1.0,3.75,,E3\r\n1.0,4.0,B4,F3\r\n1.0,4.25,,D3\r\n1.0,4.5,C5,E3\r\n1.0,4.75,,C3\r\n2.0,1.0,D5,G3\r\n2.0,1.25,G4,\r\n2.0,1.5,A4,G2\r\n2.0,1.75,B4,\r\n2.0,2.0,C5,Rest\r\n2.0,2.25,A4,\r\n2.0,2.5,B4,\r\n2.0,2.75,G4,\r\n2.0,3.0,D5,\r\n2.0,3.25,,G3\r\n2.0,3.5,G5,A3\r\n2.0,3.75,,B3\r\n2.0,4.0,F5,C4\r\n2.0,4.25,,A3\r\n2.0,4.5,G5,B3\r\n2.0,4.75,,G3\r\n3.0,1.0,E5,C4\r\n3.0,1.25,A5,\r\n3.0,1.5,G5,B3\r\n3.0,1.75,F5,\r\n3.0,2.0,E5,C4\r\n3.0,2.25,G5,\r\n3.0,2.5,F5,D4\r\n3.0,2.75,A5,\r\n3.0,3.0,G5,E4\r\n3.0,3.25,F5,\r\n3.0,3.5,E5,G3\r\n3.0,3.75,D5,\r\n3.0,4.0,C5,A3\r\n3.0,4.25,E5,\r\n3.0,4.5,D5,B3\r\n3.0,4.75,F5,\r\n4.0,1.0,E5,C4\r\n4.0,1.25,D5,\r\n4.0,1.5,C5,E3\r\n4.0,1.75,B4,\r\n4.0,2.0,A4,F#3\r\n4.0,2.25,C5,\r\n4.0,2.5,B4,G3\r\n4.0,2.75,D5,\r\n4.0,3.0,C5,A3\r\n4.0,3.25,B4,\r\n4.0,3.5,A4,B3\r\n4.0,3.75,G

get_melodic_intervals

In [6]:
mel = piece.melodic(df=nr, kind='d')
mel

1     2
Measure Beat            
1.0     1.00  Rest  Rest
        1.50     2   NaN
        1.75     2   NaN
        2.00     2   NaN
        2.25    -3   NaN
...            ...   ...
21.0    4.00    -2     2
        4.25     7   NaN
        4.50    -5    -8
        4.75     4   NaN
22.0    1.00   NaN    -5

[327 rows x 2 columns]

Save melodic intervals to excel

In [7]:
# writer = pd.ExcelWriter('saved_csv/CRIM_Model_0008.xlsx', engine = 'xlsxwriter')
# mel.to_excel(writer, sheet_name = 'CRIM Model 0008') 
# writer.close()

Melodic intervals (chromatic) to excel

In [8]:
# mel1 = piece.melodic(df=nr, kind='c')
# writer = pd.ExcelWriter('saved_csv/CRIM_Model_0009.xlsx', engine = 'xlsxwriter')
# mel1.to_excel(writer, sheet_name = 'CRIM Model 0009') 
# writer.close()

Melodic intervals into plot

In [9]:
import plotly.express as px

In [10]:
# reshape the table
mel_long = mel.reset_index()
mel_long = mel_long.melt(
    id_vars=["Measure", "Beat"],
    var_name="Voice",
    value_name="Interval"
)

# remove nan and rest
mel_long = mel_long.dropna(subset=["Interval"])
mel_long = mel_long[mel_long["Interval"] != "Rest"]

# make intervals numeric
mel_long["Interval"] = pd.to_numeric(mel_long["Interval"])

# count frequencies by interval and voice
freq = (
    mel_long
    .groupby(["Interval", "Voice"])
    .size()
    .reset_index(name="Count")
)

# plot
fig = px.bar(
    freq,
    x="Interval",
    y="Count",
    color="Voice",
    barmode="stack",
    title="Distribution of Melodic Intervals",
    labels={"Interval": "Interval", "Count": "Count"}
)
fig.show()

Chi-squared test for the null hypothesis of "Voice 1 and 2 have the same melodic-interval distribution"

In [11]:
# freq table: rows = interval, cols = voice
table = (
    mel_long.groupby(["Interval", "Voice"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
table

Voice,1,2
Interval,,
-9,0,1
-8,0,5
-7,1,1
-6,4,3
-5,4,5
-4,2,3
-3,28,22
-2,81,66
1,4,1


In [12]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)
print("dof:", dof)

Chi-square: 12.457036359496692
p-value: 0.6441573360865536
dof: 15


Cosine similarity between melodic-interval distributions for voice 1 and 2

In [13]:
import numpy as np

x = table["1"].values
y = table["2"].values

cos_sim = np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))
print("Cosine similarity:", cos_sim)

Cosine similarity: 0.9918228478287269


get_melodic_ngrams

In [14]:
n = 4
mel = piece.melodic(kind='d', end=False)

mel_ngrams = piece.ngrams(df=mel, n=n, offsets="first").fillna("")
mel_ngrams = piece.numberParts(mel_ngrams)
mel_ngrams = piece.detailIndex(mel_ngrams, offset=True)
# mel_ngrams_csv = mel_ngrams.to_csv(index=True)
# mel_ngrams

# mel_ngrams = piece.ngrams(df=mel, n=n, offsets="first")
# mel_ngrams = piece.entries(
#     df=mel_ngrams, thematic=False, anywhere=False, fermatas=True, exclude=[]
# ).fillna("")
# mel_ngrams = piece.numberParts(mel_ngrams)
# mel_ngrams = piece.detailIndex(mel_ngrams, offset=True)

def tuple_to_string(x):
    return "_".join(map(str, x)) if isinstance(x, tuple) else x

mel_ngrams = mel_ngrams.map(tuple_to_string)

Plot melodic n-gram patterns

In [15]:
all_patterns = mel_ngrams.stack()
all_patterns = all_patterns[all_patterns != ""]

top10 = (
    all_patterns.value_counts()
    .head(10)
    .reset_index()
)
top10.columns = ["Pattern", "Frequency"]

fig = px.bar(
    top10.sort_values("Frequency"),
    x="Frequency",
    y="Pattern",
    orientation="h",
    title="Top 10 melodic n-gram patterns across all voices"
)

fig.show()

In [16]:
top10_patterns = all_patterns.value_counts().head(10).index.tolist()

long_df = mel_ngrams.stack().reset_index()
long_df.columns = ["Measure", "Beat", "Offset", "Voice", "Pattern"]

long_df = long_df[long_df["Pattern"] != ""]

long_df = long_df[long_df["Pattern"].isin(top10_patterns)]

plot_df = (
    long_df.groupby(["Pattern", "Voice"])
    .size()
    .reset_index(name="Count")
)

# preserve the global top-10 order
plot_df["Pattern"] = pd.Categorical(
    plot_df["Pattern"],
    categories=top10_patterns,
    ordered=True
)
plot_df = plot_df.sort_values("Pattern")

# stacked bar chart
fig = px.bar(
    plot_df,
    x="Pattern",
    y="Count",
    color="Voice",
    title="Top 10 melodic n-gram patterns across all voices, split by voice"
)

fig.update_layout(
    barmode="stack",
    xaxis_tickangle=-45
)

fig.show()

Chi-squared test and cosine similarity between voices for all patterns

In [17]:
# long format
long_df = mel_ngrams.stack().rename("Pattern").reset_index()

# rename columns
long_df.columns = ["Measure", "Beat", "Offset", "Voice", "Pattern"]

# remove empty patterns
long_df = long_df[long_df["Pattern"] != ""]

# contingency table
count_table = pd.crosstab(long_df["Pattern"], long_df["Voice"])

chi2, p, dof, expected = chi2_contingency(count_table)

print("Chi-squared statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)

v1 = count_table.iloc[:, 0].to_numpy()
v2 = count_table.iloc[:, 1].to_numpy()

cosine_similarity = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
print("Cosine similarity:", cosine_similarity)

Chi-squared statistic: 105.99088877558758
p-value: 0.9592463319619372
Degrees of freedom: 133
Cosine similarity: 0.9272231675675927


Chi-squared test and cosine similarity between voices for globally top 10 patterns

In [18]:
# global top 10 patterns across all voices
top10_patterns = long_df["Pattern"].value_counts().head(10).index

# keep only those patterns
long_top10 = long_df[long_df["Pattern"].isin(top10_patterns)]

# contingency table for top 10 only
count_table_top10 = pd.crosstab(long_top10["Pattern"], long_top10["Voice"])

# chi-squared test
chi2, p, dof, expected = chi2_contingency(count_table_top10)

print("Chi-squared statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)

# cosine similarity for the two voices
v1 = count_table_top10.iloc[:, 0].to_numpy()
v2 = count_table_top10.iloc[:, 1].to_numpy()

cosine_similarity = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
print("Cosine similarity:", cosine_similarity)

Chi-squared statistic: 0.7777612124730554
p-value: 0.9998014347973004
Degrees of freedom: 9
Cosine similarity: 0.9932901923195109


Heat map

In [19]:
import pandas as pd
import altair as alt

n_top = 2

# long form
long_df = mel_ngrams.stack().rename("pattern").reset_index()
long_df.columns = ["Measure", "Beat", "Offset", "voice", "pattern"]

# remove blanks
long_df = long_df[long_df["pattern"] != ""].copy()

# top 2 patterns globally
top2_patterns = long_df["pattern"].value_counts().head(n_top).index.tolist()
long_df = long_df[long_df["pattern"].isin(top2_patterns)].copy()

# make sure sorting is correct
long_df["Offset"] = long_df["Offset"].astype(float)
long_df["Measure"] = long_df["Measure"].astype(float)
long_df["Beat"] = long_df["Beat"].astype(float)
long_df["voice"] = long_df["voice"].astype(str)

long_df = long_df.sort_values(["voice", "Offset"])
long_df

# start is current offset
long_df["start"] = long_df["Offset"]

# end is next offset within the same voice
long_df["end"] = long_df["start"] + n * 0.25

# convert pattern to string if needed
plot_df = long_df.copy()
plot_df["pattern"] = plot_df["pattern"].astype(str)

selector = alt.selection_point(fields=["pattern"])

heatmap = alt.Chart(plot_df).mark_bar().encode(
    x=alt.X("start:Q", title="Offset of beats"),
    x2="end:Q",
    y=alt.Y("voice:N", title="Voice", sort=None),
    color=alt.Color("pattern:N", title="Pattern"),
    opacity=alt.condition(selector, alt.value(1), alt.value(0.3)),
    tooltip=["Measure", "Beat", "voice", "pattern", "start", "end"]
).add_params(
    selector
).properties(
    width=800,
    height=250,
    title="Top " + str(n_top) + " melodic " + str(n) + "-gram patterns"
)

heatmap

alt.Chart(...)

Heat map as in SteamLit

In [20]:
import crim_intervals.visualizations as viz

# settings:
combineUnisons = True
kind = "d"
compound = False
ngram_length = 4


# find entries for model
nr = piece.notes(combineUnisons=combineUnisons)
mel = piece.melodic(df=nr, kind=kind, compound=compound, unit=0, end=False)
mod_mel_ngrams = piece.ngrams(df=mel, n=ngram_length)
mod_entry_ngrams = piece.entries(df=mel, n=ngram_length, thematic=True, anywhere=True)
mod_mel_ngrams_duration = piece.durations(df=mel, n=ngram_length, mask_df=mod_entry_ngrams)
mod_entries_stack = list(mod_entry_ngrams.stack().unique())

print(piece.metadata)

display(viz.plot_ngrams_heatmap(mod_entry_ngrams, mod_mel_ngrams_duration, 
                        selected_patterns=mod_entries_stack,
                        voices=[]))

{'title': 'Invention No. 1 in C major,  BWV 772', 'composer': 'Bach, Johann Sebastian', 'date': None}


alt.Chart(...)

In [21]:
mod_entry_ngrams

,Part-1,Part-2
0.25,"(2, 2, 2, -3)",NaN
2.25,NaN,"(2, 2, 2, -3)"
4.25,"(2, 2, 2, -3)",NaN
6.25,NaN,"(2, 2, 2, -3)"
8.25,"(-2, -2, -2, 3)",NaN
10.25,"(-2, -2, -2, 3)",NaN
12.25,"(-2, -2, -2, 3)",NaN
14.25,"(-2, -2, -2, 3)",NaN
16.25,NaN,"(2, 2, 2, -3)"
18.25,"(-2, -2, -2, 3)",NaN


In [22]:
mod_mel_ngrams_duration

,Part-1,Part-2
0.25,1.0,NaN
2.25,NaN,1.00
4.25,1.0,NaN
6.25,NaN,1.00
8.25,1.0,NaN
10.25,1.0,NaN
12.25,1.0,NaN
14.25,1.0,NaN
16.25,NaN,1.00
18.25,1.0,NaN


In [23]:
first_occurrence = {}

for row in mod_entry_ngrams.index:
    for col in mod_entry_ngrams.columns:
        pattern = mod_entry_ngrams.loc[row, col]

        if isinstance(pattern, tuple) and pattern not in first_occurrence:
            first_occurrence[pattern] = {
                "row": row,
                "column": col,
                "duration": mod_mel_ngrams_duration.loc[row, col]
            }

result_df = (
    pd.DataFrame.from_dict(first_occurrence, orient="index")
    .reset_index()
    .rename(columns={"index": "pattern"})
)

result_df

,level_0,level_1,level_2,level_3,row,column,duration
0,2,2,2,-3,0.25,Part-1,1.0
1,-2,-2,-2,3,8.25,Part-1,1.0


In [24]:
import re
import verovio
from IPython.display import HTML, display

def play_mei_excerpt(mei_path, start_q, end_q, bpm=120):
    with open(mei_path, "r", encoding="utf-8") as f:
        mei_text = f.read()

    # Set or inject tempo
    if 'midi.bpm="' in mei_text:
        mei_text = re.sub(
            r'midi\.bpm="\d+(\.\d+)?"',
            f'midi.bpm="{bpm}"',
            mei_text,
            count=1
        )
    else:
        mei_text = re.sub(
            r'(<measure\b[^>]*>)',
            rf'\1\n  <tempo midi.bpm="{bpm}">♩ = {bpm}</tempo>',
            mei_text,
            count=1
        )

    tk = verovio.toolkit()
    tk.loadData(mei_text)
    midi_b64 = tk.renderToMIDI()

    start_sec = start_q * 60 / bpm
    end_sec = end_q * 60 / bpm

    html = f"""
    <script type="module" src="https://cdn.jsdelivr.net/npm/html-midi-player@1.6.0/+esm"></script>

    <div style="margin:10px 0;">
      <midi-player id="player"
                   src="data:audio/midi;base64,{midi_b64}"
                   sound-font>
      </midi-player>
    </div>

    <button id="play_excerpt_btn" disabled>Play pattern</button>
    <span id="status" style="margin-left:10px;">Loading player...</span>

    <script>
    (function() {{
        const player = document.getElementById("player");
        const btn = document.getElementById("play_excerpt_btn");
        const status = document.getElementById("status");

        const startSec = {start_sec};
        const endSec = {end_sec};

        player.addEventListener("load", () => {{
            btn.disabled = false;
            status.textContent = "Ready";
            console.log("MIDI player loaded");
        }});

        player.addEventListener("start", () => {{
            console.log("Playback started at", player.currentTime);
        }});

        player.addEventListener("stop", () => {{
            console.log("Playback stopped at", player.currentTime);
        }});

        btn.addEventListener("click", async () => {{
            try {{
                status.textContent = "Playing...";
                player.stop();
                player.currentTime = startSec;
                player.start();

                const timer = setInterval(() => {{
                    if (!player.playing || player.currentTime >= endSec) {{
                        player.stop();
                        clearInterval(timer);
                        status.textContent = "Stopped";
                    }}
                }}, 50);
            }} catch (err) {{
                console.error(err);
                status.textContent = "Error: " + err;
            }}
        }});
    }})();
    </script>
    """
    display(HTML(html))

In [ ]:

# play_mei_excerpt(mei_path, result_df.loc("0"), 1.25, bpm=60)